In [1]:
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, 'w') as f:
            f.write(text_data)
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

file_path = "datasets/instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(file_path, url)
print(len(data))
            

1100


In [2]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry['input'] else ""
    )
    return instruction_text + input_text

model_input = format_input(data[999])
desired_output = data[999]['output']
print(model_input + desired_output)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?An antonym of 'complicated' is 'simple'.


In [3]:
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.10)
valid_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
valid_data = data[train_portion + test_portion:]
print(f"Train data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")
print(f"Validation data size: {len(valid_data)}")

Train data size: 935
Test data size: 110
Validation data size: 55


In [4]:
import torch
from torch.utils.data import Dataset
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.encoded_texts[idx]

## 自定义批处理聚合函数

In [5]:
def custom_collate_fn(
        batch,
        pad_token_id=50256,
        ignore_index=-100,
        allowed_max_length=None,
        device="cpu"
):
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst = []
    target_lst = []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] * (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            targets = targets[:allowed_max_length]
            inputs = inputs[:allowed_max_length]
        inputs_lst.append(inputs)
        target_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(target_lst).to(device)
    return inputs_tensor, targets_tensor

In [6]:
input1 = [0, 1, 2, 3, 4]
input2 = [5, 6, 7]
input3 = [9, 10]

batch = (
    input1,
    input2,
    input3
)
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6,     7, 50256, 50256],
        [    9,    10, 50256, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6,     7, 50256,  -100,  -100],
        [   10, 50256,  -100,  -100,  -100]])


## 探索为什么-100

In [7]:
logit1 = torch.tensor([[-1.0, 1.0],
                       [-0.5, 1.5],
                       [-0.5, 1.5]])
target1 = torch.tensor([0, 1, -100])
loss1 = torch.nn.functional.cross_entropy(logit1, target1)
print(loss1)

tensor(1.1269)


## 创建数据loader

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from functools import partial
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)


In [9]:
from torch.utils.data import DataLoader
import tiktoken
num_workers = 0
batch_size = 8
tokenizer = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer=tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
valid_dataset = InstructionDataset(valid_data, tokenizer=tokenizer)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)
test_dataset = InstructionDataset(test_data, tokenizer=tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    collate_fn=customized_collate_fn,
    drop_last=True
)


In [10]:
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.Size([8, 68])


## 加载预训练模型

In [11]:
from gpt_download import download_and_load_gpt2
from models.GPTModel import GPTModel
from utils import load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "dropout": 0.1,
    "bias": True,
}
model_config = {
    "gpt2-small" : {"emb_dim":768, "num_layers":12, "num_heads":12},
    "gpt2-medium" : {"emb_dim":1024, "num_layers":24, "num_heads":16},
    "gpt2-large" : {"emb_dim":1280, "num_layers":36, "num_heads":20},
    "gpt2-xl" : {"emb_dim":1600, "num_layers":48, "num_heads":25}
}

BASE_CONFIG.update(model_config["gpt2-medium"])
settings, params = download_and_load_gpt2(
    model_size="355M",
    models_dir="../models/"
)
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

ModuleNotFoundError: No module named 'tensorflow'